# Распознавание кода Морзе из аудиосигнала (v10.x Refactored)

Этот Jupyter Notebook представляет собой основной конвейер для обучения и оценки модели распознавания кода Морзе.

**Структура ноутбука:**

1.  **Импорты:** Загрузка всех необходимых библиотек.
2.  **Определение Корня Проекта:** Автоматическое определение базовой директории проекта.
3.  **Загрузка Конфигурации:** Чтение базового файла конфигурации (`experiment_base.json`).
4.  **Обновление Конфигурации и Утилиты:** Динамическое обновление конфига (устройство, калибровка), установка seed, настройка MLflow, очистка CUDA.
5.  **Подготовка Данных:** Загрузка CSV-файлов, создание словарей символов, формирование путей к аудио. Сканирование карт Перлина.
6.  **Разделение Данных:** Разделение основного датасета на обучающую и валидационную выборки.
7.  **Создание Аугментатора:** Инициализация объекта для аудио-аугментаций.
8.  **Основной Пайплайн Выполнения:** Оркестрация процессов обучения, дообучения (если применимо) и генерации предсказаний (submission). Включает запуск/остановку MLflow run, сохранение конфигураций запуска.

---

## 1. Импорты
---

In [1]:
# Ячейка 1: Импорты (Чисто, сгруппировано)
# ----------------------------------------

# Стандартные библиотеки Python
import os
import gc
import sys
import json
import random
import time
import warnings
import math
import traceback
import contextlib
import re
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# Подавление ошибки Intel MKL (если необходимо)
os.environ['KMP_DUPLICATE_LIB_OK']='True'

# Основные библиотеки Data Science
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import Levenshtein # pip install python-Levenshtein

# PyTorch и связанные библиотеки
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torch.nn.utils.rnn import pad_sequence
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import ReduceLROnPlateau, OneCycleLR
import torchaudio # Для SpecAugment и др.

# Аугментации Аудио
import audiomentations # pip install audiomentations

# Утилиты и Логирование
from tqdm.notebook import tqdm # Или from tqdm import tqdm для скриптов
import mlflow # pip install mlflow

# Игнорирование предупреждений (использовать с осторожностью)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# --- Вывод версий для воспроизводимости ---
print("--- Версии ключевых библиотек ---")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Torchaudio: {torchaudio.__version__}")
print(f"Librosa: {librosa.__version__}")
print(f"Audiomentations: {audiomentations.__version__}")
print(f"Numpy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"MLflow: {mlflow.__version__}")
print(f"Levenshtein: {'Доступен' if 'Levenshtein' in sys.modules else 'Не найден!'}")
print("-" * 30)
print("Ячейка 1 (Импорты) выполнена.")

--- Версии ключевых библиотек ---
Python: 3.9.21
PyTorch: 2.5.1+cu121
Torchaudio: 2.5.1+cu121
Librosa: 0.10.2.post1
Audiomentations: 0.40.0
Numpy: 2.0.2
Pandas: 2.2.3
MLflow: 2.21.3
Levenshtein: Доступен
------------------------------
Ячейка 1 (Импорты) выполнена.


## 2. Определение Корня Проекта

Эта ячейка определяет корневую директорию проекта (`PROJECT_ROOT`) и добавляет ее в `sys.path`, чтобы можно было импортировать модули из папки `src`. Логика пытается найти корень, проверяя наличие папки `src` на текущем или родительском уровне.

---


In [2]:
# Ячейка 2: Определение Корня Проекта и Добавление в sys.path
# ---------------------------------------------------------

try:
    # Пытаемся определить корень проекта
    current_working_dir = Path(os.getcwd()).resolve() # Получаем абсолютный путь
    PROJECT_ROOT = None

    # Сценарий 1: Мы в корне проекта (есть папка src)
    if (current_working_dir / 'src').is_dir():
        PROJECT_ROOT = current_working_dir
        print(f"Обнаружен корень проекта (содержит 'src'): {PROJECT_ROOT}")
    # Сценарий 2: Мы в папке 'notebooks' (папка 'src' на уровень выше)
    elif (current_working_dir.parent / 'src').is_dir():
        PROJECT_ROOT = current_working_dir.parent
        print(f"Обнаружен корень проекта (родитель содержит 'src'): {PROJECT_ROOT}")
    else:
        # Запасной вариант: предполагаем, что мы на один уровень ниже корня
        PROJECT_ROOT = current_working_dir.parent
        print(f"Предупреждение: Не удалось надежно определить корень проекта по папке 'src'.")
        print(f"Предполагаемый корень (родитель текущей папки): {PROJECT_ROOT}")
        # Добавим проверку на существование базовых файлов/папок, если нужно
        if not (PROJECT_ROOT / 'README.md').exists() and not (PROJECT_ROOT / 'config').exists():
             print(f"!!! ВНИМАНИЕ: Предполагаемый корень {PROJECT_ROOT} может быть неверным!")

    # Добавляем корень проекта в sys.path, если его там еще нет
    project_root_str = str(PROJECT_ROOT)
    if project_root_str not in sys.path:
        sys.path.insert(0, project_root_str) # Добавляем в начало
        print(f"Добавлен '{project_root_str}' в sys.path.")
    else:
        print(f"'{project_root_str}' уже находится в sys.path.")

except Exception as e_proj_root:
     print(f"❌ КРИТИЧЕСКАЯ ОШИБКА при определении корня проекта: {e_proj_root}")
     print("   Дальнейшее выполнение может быть невозможным.")
     raise # Прерываем выполнение, так как без корня не импортировать src

print("\n--- Ячейка 2 (Определение Корня Проекта) завершена ---")
# Импорты из src 
from src.utils.common import set_seed
from src.utils.mlflow_utils import setup_mlflow
from src.training.pipeline import run_training
from src.data_processing.text import create_char_map
from src.utils.path_utils import create_full_path
from src.utils.file_system import find_available_perlin_maps
from src.data_processing.augmentations import create_audio_augmenter
from src.inference.predict import generate_submission

Обнаружен корень проекта (родитель содержит 'src'): C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder
Добавлен 'C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder' в sys.path.

--- Ячейка 2 (Определение Корня Проекта) завершена ---


## 3. Загрузка Базовой Конфигурации

Загружаем основной файл конфигурации (`experiment_base.json` или аналогичный) из папки `config`. Этот файл содержит статические параметры эксперимента.

---

In [3]:
# Ячейка 3: Загрузка Базовой Конфигурации
# ---------------------------------------

CONFIG = {} # Словарь для хранения конфигурации
CONFIG_PATH = PROJECT_ROOT / 'config' / 'experiment_base.json' # Путь к базовому конфигу

print(f"Попытка загрузки базовой конфигурации из: {CONFIG_PATH}")

try:
    if not CONFIG_PATH.is_file():
        raise FileNotFoundError(f"Файл конфигурации не найден: {CONFIG_PATH}")

    with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
        CONFIG = json.load(f)
    print(f"Базовая конфигурация ({CONFIG_PATH.name}) успешно загружена.")

    # Проверка наличия ключевых секций (пример)
    required_sections = ["paths", "audio", "model", "training", "ctc", "random_seed"]
    for section in required_sections:
        if section not in CONFIG:
            warnings.warn(f"Секция '{section}' отсутствует в базовой конфигурации!")

except FileNotFoundError as e:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА: {e}")
    print("   Убедитесь, что файл конфигурации существует и путь к нему верен.")
    raise
except json.JSONDecodeError as e:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА: Не удалось разобрать JSON файл конфигурации: {e}")
    print(f"   Проверьте синтаксис файла: {CONFIG_PATH}")
    raise
except Exception as e:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА при загрузке конфигурации: {e}")
    traceback.print_exc(limit=1)
    raise

print("\n--- Ячейка 3 (Загрузка Конфигурации) завершена ---")

Попытка загрузки базовой конфигурации из: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\config\experiment_base.json
Базовая конфигурация (experiment_base.json) успешно загружена.

--- Ячейка 3 (Загрузка Конфигурации) завершена ---


## 4. Обновление Конфигурации и Настройка Утилит

Эта ячейка выполняет несколько задач:
1.  **Динамическое обновление `CONFIG`:** Устанавливает устройство (`cuda`/`cpu`), вычисляет `freq_dim`, обрабатывает режим калибровки.
2.  **Установка Random Seed:** Фиксирует генераторы случайных чисел для воспроизводимости.
3.  **Настройка MLflow:** Инициализирует MLflow согласно конфигурации.
4.  **Очистка CUDA Cache:** Освобождает память на GPU перед началом работы.

---

In [4]:
# Ячейка 4: Обновление Конфигурации и Настройка Утилит (Исправлен вызов setup_mlflow)
# ---------------------------------------------------------------------------------

IS_MLFLOW_ACTIVE = False # Глобальный флаг статуса MLflow

try:
    # --- 1. Динамическое обновление CONFIG ---
    print("--- Обновление конфигурации ---")
    # Определение устройства
    if CONFIG.get("device", "auto") == "auto":
        CONFIG["device"] = "cuda" if torch.cuda.is_available() else "cpu"
    elif CONFIG["device"] == "cuda" and not torch.cuda.is_available():
        print("Предупреждение: Устройство в конфиге 'cuda', но CUDA недоступна. Используется 'cpu'.")
        CONFIG["device"] = "cpu"
    print(f"Используемое устройство: {CONFIG['device']}")

    # Вычисление model.freq_dim
    try:
        n_fft = CONFIG.get("audio", {}).get("n_fft")
        if n_fft is not None:
            # Убедимся, что секция model существует
            if "model" not in CONFIG: CONFIG["model"] = {}
            CONFIG["model"]["freq_dim"] = n_fft // 2 + 1
            print(f"Вычислено model.freq_dim: {CONFIG['model']['freq_dim']} (из n_fft={n_fft})")
        else:
            warnings.warn("Не найден 'audio.n_fft' в конфиге для вычисления 'model.freq_dim'.")
    except KeyError as e:
        print(f"❌ ОШИБКА: Отсутствует ключ '{e}' при доступе к CONFIG['model'].")
        raise

    # Обработка режима калибровки
    if CONFIG.get("mode", {}).get("calibration_mode", False):
        print("\n!!! РЕЖИМ КАЛИБРОВКИ АКТИВЕН !!!")
        cal_epochs = CONFIG["mode"].get("calibration_epochs", 1)
        cal_patience = max(1, cal_epochs // 2)
        for stage in ["training", "finetuning"]:
            if stage in CONFIG:
                # Убедимся, что stage является словарем
                if not isinstance(CONFIG[stage], dict): CONFIG[stage] = {}
                CONFIG[stage]["epochs"] = cal_epochs
                CONFIG[stage]["early_stopping_patience"] = cal_patience
        print(f"  Эпохи (Train/Finetune): {cal_epochs}")
        print(f"  Patience (Train/Finetune): {cal_patience}")

    # --- 2. Установка Random Seed ---
    seed = CONFIG.get("random_seed")
    if seed is None:
        warnings.warn("Ключ 'random_seed' не найден в CONFIG. Воспроизводимость не гарантирована.")
    else:
        print(f"\n--- Установка Random Seed: {seed} ---")
        set_seed(seed)
        print("Seed установлен для Python, Numpy и PyTorch.")

    # --- 3. Настройка MLflow ---
    print("\n--- Настройка MLflow ---")
    # Вызываем setup_mlflow БЕЗ аргумента project_root, так как функция его не ожидает
    # Функция должна сама брать нужные пути (например, tracking_uri) из CONFIG
    IS_MLFLOW_ACTIVE = setup_mlflow(CONFIG) # <--- ИСПРАВЛЕННЫЙ ВЫЗОВ
    print(f"Статус MLflow после настройки: {'Активен' if IS_MLFLOW_ACTIVE else 'Неактивен'}")

    # --- 4. Очистка CUDA Cache ---
    if CONFIG.get('device') == 'cuda': # Используем .get() для безопасности
        print("\n--- Очистка кэша CUDA ---")
        torch.cuda.empty_cache()
        gc.collect() # Добавим сборку мусора Python для надежности
        print("Кэш CUDA очищен.")

except KeyError as ke:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА: Ключ '{ke}' не найден в CONFIG.")
    print("   Проверьте ваш файл experiment_base.json.")
    traceback.print_exc(limit=1) # Покажем место ошибки в конфиге
    raise
except NameError as ne:
     print(f"❌ КРИТИЧЕСКАЯ ОШИБКА: Переменная не определена - {ne}. Выполните предыдущие ячейки.")
     raise
except TypeError as te: # Ловим конкретно TypeError, если вдруг сигнатура опять не совпадет
     print(f"❌ КРИТИЧЕСКАЯ ОШИБКА TypeError при вызове функции (возможно, setup_mlflow): {te}")
     print("   Проверьте аргументы, передаваемые в функции из src.")
     traceback.print_exc(limit=1)
     raise
except Exception as e_cell4:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА в Ячейке 4: {e_cell4}")
    traceback.print_exc(limit=2)
    raise

print("\n--- Ячейка 4 (Обновление Конфигурации и Утилиты) завершена ---")

2025/04/23 10:28:13 INFO mlflow.tracking.fluent: Experiment with name 'MorseCodeRecognition_v10' does not exist. Creating a new experiment.


--- Обновление конфигурации ---
Используемое устройство: cuda
Вычислено model.freq_dim: 257 (из n_fft=512)

--- Установка Random Seed: 42 ---
Seed установлен: 42
Seed установлен для Python, Numpy и PyTorch.

--- Настройка MLflow ---
MLflow Tracking URI не указан, используется локальное логирование (папка 'mlruns').
MLflow Experiment 'MorseCodeRecognition_v10' установлен.
✅ MLflow успешно настроен и активен.
Статус MLflow после настройки: Активен

--- Очистка кэша CUDA ---
Кэш CUDA очищен.

--- Ячейка 4 (Обновление Конфигурации и Утилиты) завершена ---


## 5. Подготовка Данных

Загрузка метаданных из CSV-файлов (`train.csv`, `sample_submission.csv`), создание словарей для преобразования символов в индексы и обратно, формирование полных путей к аудиофайлам. Также здесь происходит сканирование доступных карт шума Перлина, если они используются для аугментации.

---

In [5]:
# --- Глобальные переменные для данных ---
train_df_full = pd.DataFrame()
test_df = pd.DataFrame()
char_to_int, int_to_char = {}, {}
vocab_size = 0
AVAILABLE_PERLIN_INDICES = [] # Список доступных индексов карт Перлина

try:
    # --- Получение параметров из CONFIG ---
    print("--- Загрузка параметров путей и столбцов из CONFIG ---")
    TRAIN_FILE_COLUMN = CONFIG['train_file_column']
    TEST_FILE_COLUMN = CONFIG['test_file_column']
    MORSE_CODE_COLUMN = CONFIG['morse_code_column']
    DATA_ROOT_STR = CONFIG.get('paths', {}).get('data_dir', 'data')
    AUDIO_FOLDER_NAME = CONFIG.get('paths', {}).get('audio_folder_name', 'morse_dataset/morse_dataset')
    PERLIN_MAPS_DIR_STR = CONFIG.get("paths", {}).get("generic_perlin_maps_dir")

    # --- Формирование путей к данным ---
    data_root_path = PROJECT_ROOT / DATA_ROOT_STR
    raw_data_path = data_root_path / 'raw'
    train_csv_path = raw_data_path / "train.csv"
    test_csv_path = raw_data_path / "sample_submission.csv"
    audio_folder_path = raw_data_path / AUDIO_FOLDER_NAME

    print(f"Ожидаемый путь к сырым данным: {raw_data_path.resolve()}")
    print(f"Ожидаемый путь к train.csv: {train_csv_path.resolve()}")
    print(f"Ожидаемый путь к sample_submission.csv: {test_csv_path.resolve()}")
    print(f"Ожидаемый путь к аудиофайлам: {audio_folder_path.resolve()}")

    # --- Загрузка CSV ---
    print("\n--- Загрузка CSV файлов ---")
    if not train_csv_path.is_file(): raise FileNotFoundError(f"Файл train.csv не найден: {train_csv_path}")
    if not test_csv_path.is_file(): raise FileNotFoundError(f"Файл sample_submission.csv не найден: {test_csv_path}")

    train_df_full = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)
    print(f"Загружено: train.csv ({len(train_df_full)} строк), sample_submission.csv ({len(test_df)} строк)")

    required_train_cols = [TRAIN_FILE_COLUMN, MORSE_CODE_COLUMN]
    required_test_cols = [TEST_FILE_COLUMN]
    if not all(col in train_df_full.columns for col in required_train_cols):
        raise ValueError(f"Столбцы {required_train_cols} должны быть в train.csv!")
    if not all(col in test_df.columns for col in required_test_cols):
        raise ValueError(f"Столбец {TEST_FILE_COLUMN} должен быть в sample_submission.csv!")

    # --- Создание словарей символов ---
    print("\n--- Создание словаря символов ---")
    char_to_int, int_to_char, vocab_size = create_char_map(
        train_df_full[MORSE_CODE_COLUMN].astype(str).tolist(),
        CONFIG["ctc"]
    )
    # Убедимся, что секция model существует перед добавлением vocab_size
    if "model" not in CONFIG: CONFIG["model"] = {}
    CONFIG["model"]["vocab_size"] = vocab_size
    print(f"Словарь создан. Размер (включая спец. символы): {vocab_size}")

    # --- Формирование полных путей к аудиофайлам ---
    print("\n--- Формирование полных путей к аудиофайлам ---")
    if not audio_folder_path.is_dir():
         warnings.warn(f"Папка с аудио {audio_folder_path} не найдена! Проверьте путь и конфиг.")

    train_df_full['full_path'] = train_df_full[TRAIN_FILE_COLUMN].apply(lambda x: create_full_path(x, audio_folder_path))
    test_df['full_path'] = test_df[TEST_FILE_COLUMN].apply(lambda x: create_full_path(x, audio_folder_path))
    print("Полные пути добавлены в DataFrame'ы.")

    # --- Сканирование карт Перлина ---
    print("\n--- Сканирование доступных карт Перлина ---")
    if PERLIN_MAPS_DIR_STR:
        perlin_maps_dir = PROJECT_ROOT / PERLIN_MAPS_DIR_STR
        print(f"Поиск карт в: {perlin_maps_dir.resolve()}")
        AVAILABLE_PERLIN_INDICES = find_available_perlin_maps(perlin_maps_dir) # Функция должна вернуть список индексов
        if not AVAILABLE_PERLIN_INDICES:
            print("Предупреждение: Не найдено доступных карт Перлина.")
            if CONFIG.get("perlin_augmentation", {}).get("apply", False):
                print("   Аугментация Perlin будет отключена.")
                if "perlin_augmentation" not in CONFIG: CONFIG["perlin_augmentation"] = {}
                CONFIG["perlin_augmentation"]["apply"] = False
        else:
            num_maps = len(AVAILABLE_PERLIN_INDICES)
            print(f"Найдено {num_maps} карт Перлина.")
            # --- ИСПРАВЛЕНИЕ: Используем setdefault ---
            # Если CONFIG['info'] нет, создаем его как пустой словарь {},
            # затем добавляем ключ 'num_available_perlin_maps'.
            CONFIG.setdefault("info", {})["num_available_perlin_maps"] = num_maps
            print(f"Количество карт ({num_maps}) сохранено в CONFIG['info']['num_available_perlin_maps']")
            # --- КОНЕЦ ИСПРАВЛЕНИЯ ---
    else:
        print("Путь к картам Перлина ('generic_perlin_maps_dir') не указан в конфиге.")
        if CONFIG.get("perlin_augmentation", {}).get("apply", False):
            print("   Аугментация Perlin будет отключена.")
            if "perlin_augmentation" not in CONFIG: CONFIG["perlin_augmentation"] = {}
            CONFIG["perlin_augmentation"]["apply"] = False

except FileNotFoundError as e: print(f"❌ ОШИБКА: Файл не найден - {e}"); raise
except ValueError as e: print(f"❌ ОШИБКА: Некорректное значение или столбец - {e}"); raise
except KeyError as e:
    print(f"❌ ОШИБКА: Ключ не найден в CONFIG - {e}")
    print("   Проверьте ваш файл experiment_base.json или логику обновления CONFIG.")
    traceback.print_exc(limit=1) # Покажем место ошибки
    raise
except Exception as e_cell5:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА в Ячейке 5: {e_cell5}")
    traceback.print_exc(limit=2)
    raise

print("\n--- Ячейка 5 (Подготовка Данных) завершена ---")

--- Загрузка параметров путей и столбцов из CONFIG ---
Ожидаемый путь к сырым данным: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\data\raw
Ожидаемый путь к train.csv: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\data\raw\train.csv
Ожидаемый путь к sample_submission.csv: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\data\raw\sample_submission.csv
Ожидаемый путь к аудиофайлам: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\data\raw\morse_dataset\morse_dataset

--- Загрузка CSV файлов ---
Загружено: train.csv (30000 строк), sample_submission.csv (5000 строк)

--- Создание словаря символов ---
Найдено уникальных символов в текстах (44):  #0123456789АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ
Размер словаря (Vocab Size, включая бланк): 45
Словарь (idx: char): {0: '<blank>', 1: ' ', 2: '#', 3: '0', 4: '1', 5: '2', 6: '3', 7: '4', 8: '5', 9: '6', 10: '7', 11: '8', 12: '9', 13: 'А', 14: 'Б', 15: 'В', 16: 'Г', 17: 'Д', 18: 'Е', 19: 'Ж', 20: 'З', 21: 'И', 22: 'Й', 23: 

## 6. Разделение Данных на Обучение и Валидацию

Разделяем загруженный `train_df_full` на обучающий (`train_split_df`) и валидационный (`val_split_df`) наборы данных в соответствии с пропорцией `val_split_ratio` из конфигурации. Используем фиксированный `random_seed` для воспроизводимости разделения. Учитывается режим калибровки для использования подмножества данных.

---

In [6]:
# Ячейка 6: Разделение Данных на Обучение и Валидацию
# -------------------------------------------------

train_split_df, val_split_df = None, None # Инициализация

try:
    print("--- Разделение данных на Train/Validation ---")
    base_df = train_df_full # DataFrame из предыдущей ячейки
    working_df = None

    # --- Обработка режима калибровки ---
    if CONFIG.get("mode", {}).get("calibration_mode", False):
         subset_size = CONFIG["mode"].get("calibration_subset_size", 100)
         if subset_size < len(base_df):
             print(f"Режим калибровки: Используется случайное подмножество из {subset_size} записей.")
             # Используем seed из конфига для воспроизводимости выборки
             working_df = base_df.sample(n=subset_size, random_state=CONFIG["random_seed"]).copy().reset_index(drop=True)
         else:
             print(f"Режим калибровки: Размер подмножества ({subset_size}) >= размеру данных ({len(base_df)}). Используется полный набор.")
             working_df = base_df.copy().reset_index(drop=True)
    else: # Обычный режим
        working_df = base_df.copy().reset_index(drop=True)

    # --- Выполнение разделения ---
    val_split_ratio = CONFIG.get("training", {}).get("val_split_ratio", 0.1)
    if not (0 < val_split_ratio < 1):
        raise ValueError(f"Некорректный val_split_ratio ({val_split_ratio}). Должен быть между 0 и 1.")

    val_size = int(len(working_df) * val_split_ratio)
    train_size = len(working_df) - val_size

    if val_size <= 0 or train_size <= 0:
        raise ValueError(f"Некорректные размеры после разделения: Train={train_size}, Val={val_size}. Проверьте val_split_ratio и размер данных.")

    print(f"Разделение {len(working_df)} записей на Train ({train_size}) и Val ({val_size}) с Seed: {CONFIG['random_seed']}")

    # Используем генератор PyTorch для воспроизводимого разделения
    generator = torch.Generator().manual_seed(CONFIG["random_seed"])
    train_indices, val_indices = random_split(range(len(working_df)), [train_size, val_size], generator=generator)

    # Создаем финальные DataFrame'ы
    train_split_df = working_df.iloc[train_indices.indices].copy().reset_index(drop=True)
    val_split_df = working_df.iloc[val_indices.indices].copy().reset_index(drop=True)

    print(f"Данные успешно разделены: Train={len(train_split_df)}, Val={len(val_split_df)}")
    # print("\nПример Train Split:")
    # print(train_split_df.head(2))
    # print("\nПример Val Split:")
    # print(val_split_df.head(2))

except NameError as ne: print(f"❌ ОШИБКА: Переменная 'train_df_full' не определена. Выполните Ячейку 5. {ne}"); raise
except ValueError as ve: print(f"❌ ОШИБКА при разделении данных: {ve}"); raise
except KeyError as ke: print(f"❌ ОШИБКА: Ключ не найден в CONFIG - {ke}"); raise
except Exception as e_split:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА на этапе разделения данных: {e_split}")
    traceback.print_exc(limit=2)
    raise # Прерываем, так как без разделения обучение невозможно

print("\n--- Ячейка 6 (Разделение Данных) завершена ---")

--- Разделение данных на Train/Validation ---
Разделение 30000 записей на Train (27000) и Val (3000) с Seed: 42
Данные успешно разделены: Train=27000, Val=3000

--- Ячейка 6 (Разделение Данных) завершена ---


## 7. Создание Аудио-Аугментатора

Инициализируем объект для применения аудио-аугментаций (если `audio_augmentation.apply` установлен в `True` в конфигурации). Используется функция `create_audio_augmenter` из `src`.

---

In [7]:
# Ячейка 7: Создание Аудио-Аугментатора
# ------------------------------------

AUDIO_AUGMENTER_GLOBAL = None # Глобальный объект аугментатора

try:
    print("--- Создание объекта аудио-аугментатора ---")
    if CONFIG.get("audio_augmentation", {}).get("apply", False):
        # Функция create_audio_augmenter должна быть в src/data_processing/augmentations.py
        # Она читает параметры аугментаций из секции "audio_augmentation" в CONFIG
        AUDIO_AUGMENTER_GLOBAL = create_audio_augmenter(CONFIG)
        if AUDIO_AUGMENTER_GLOBAL:
            print("Глобальный объект аудио-аугментатора успешно создан.")
            # print(AUDIO_AUGMENTER_GLOBAL.transforms) # Показать список трансформаций
        else:
            # Это может произойти, если pipeline аугментаций оказался пустым
            print("Предупреждение: Аудио-аугментатор не создан (возможно, pipeline пуст).")
            CONFIG["audio_augmentation"]["apply"] = False # Убедимся, что флаг отключен
    else:
        print("Аудио-аугментации НЕ будут применяться (apply=False в конфиге).")

except KeyError as ke: print(f"❌ ОШИБКА: Ключ не найден в CONFIG при создании аугментатора - {ke}"); raise
except Exception as e_aug:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА при создании аудио-аугментатора: {e_aug}")
    traceback.print_exc(limit=2)
    # Не прерываем выполнение, но аугментации не будут работать
    AUDIO_AUGMENTER_GLOBAL = None
    CONFIG["audio_augmentation"]["apply"] = False # Отключаем флаг

print("\n--- Ячейка 7 (Создание Аугментатора) завершена ---")

--- Создание объекта аудио-аугментатора ---
Аудио-аугментации НЕ будут применяться (apply=False в конфиге).

--- Ячейка 7 (Создание Аугментатора) завершена ---


## 8. Основной Пайплайн Выполнения

Эта ячейка является сердцем ноутбука. Она оркестрирует весь процесс:
1.  **Подготовка к запуску:** Определяет режим работы, создает директорию для результатов этого конкретного запуска, сохраняет начальную конфигурацию.
2.  **Запуск MLflow Run:** Стартует новый эксперимент в MLflow (если MLflow активен).
3.  **Основное Обучение (`train`):** Запускает этап обучения с нуля (если режим позволяет).
4.  **Дообучение (`finetune`):** Запускает этап дообучения на основе лучшей модели из предыдущего этапа (если режим позволяет и флаг включен).
5.  **Генерация Submission:** Создает файл с предсказаниями для тестового набора данных, используя лучшую модель по итогам всех этапов.
6.  **Завершение:** Сохраняет финальную конфигурацию с результатами пайплайна, логирует артефакты в MLflow и завершает MLflow run.

---


### Ячейка 8.1: Подготовка к Запуску и Старт MLflow

Определяем режим работы, создаем уникальную директорию для результатов этого запуска, сохраняем начальную конфигурацию и инициализируем MLflow run (если он активен).


In [8]:
# Ячейка 8.1: Подготовка к Запуску и Старт MLflow
# -----------------------------------------------

# --- Проверка наличия необходимых переменных ---
required_vars = ['CONFIG', 'PROJECT_ROOT', 'train_split_df', 'val_split_df',
                 'test_df', 'char_to_int', 'int_to_char', 'AUDIO_AUGMENTER_GLOBAL',
                 'AVAILABLE_PERLIN_INDICES', 'IS_MLFLOW_ACTIVE', 'sys', 'time',
                 'Path', 'json', 'mlflow', 'torch', 'np', 'contextlib', 'traceback']
for var in required_vars:
    if var not in locals() and var not in sys.modules: # Проверяем и переменные, и модули
        raise NameError(f"Переменная или модуль '{var}' не определен(а). Выполните предыдущие ячейки, особенно Ячейку 1.")

# --- Определение режима работы и директории запуска ---
run_mode = CONFIG.get("run_mode", "train_and_finetune")
finetune_apply_flag = CONFIG.get("finetuning", {}).get("apply", False)

run_description_safe = CONFIG.get('run_description', 'default_run').replace(":", "-").replace(" ", "_").replace("/", "_")
run_dir_name = run_description_safe
OUTPUT_DIR_RUN = PROJECT_ROOT / CONFIG["paths"]["output_dir"] / run_dir_name

# --- Инициализация переменных для результатов ---
final_model_path = ""
final_levenshtein = float('inf')
pipeline_start_time = time.time()
active_mlflow_run_id = None

print("\n" + "="*50)
print(f" ЗАПУСК ОСНОВНОГО КОНВЕЙЕРА (Режим: {run_mode}) ")
print(f" Директория для результатов этого запуска: {OUTPUT_DIR_RUN.resolve()}")
print("="*50)

try:
    # --- Создание выходной директории ЗАПУСКА ---
    OUTPUT_DIR_RUN.mkdir(parents=True, exist_ok=True)
    print(f"Выходная директория ЗАПУСКА создана/проверена.")

    # --- Сохранение НАЧАЛЬНОГО конфига ЗАПУСКА ---
    initial_config_save_path = OUTPUT_DIR_RUN / f"config_initial_{run_dir_name}.json"
    initial_config_snapshot = CONFIG.copy()
    initial_config_snapshot['execution_info'] = {
        'python_version': sys.version,
        'torch_version': torch.__version__,
        'start_time_utc': time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
        'project_root': str(PROJECT_ROOT),
        'output_dir_run': str(OUTPUT_DIR_RUN)
    }
    with open(initial_config_save_path, 'w', encoding='utf-8') as f:
        json.dump(initial_config_snapshot, f, indent=4, ensure_ascii=False, default=str)
    print(f"Начальная конфигурация ЗАПУСКА сохранена: {initial_config_save_path.name}")

    # --- Старт MLflow Run ---
    if IS_MLFLOW_ACTIVE:
        print("\n--- Старт MLflow Run ---")
        try:
            active_run = mlflow.start_run(run_name=run_description_safe)
            active_mlflow_run_id = active_run.info.run_id
            print(f"MLflow Run начат. ID: {active_mlflow_run_id}")
            mlflow.log_artifact(str(initial_config_save_path), artifact_path="config")
            params_to_log = {
                "run_mode": run_mode,
                "model_arch": CONFIG.get("model", {}).get("architecture", "N/A"),
                "n_fft": CONFIG.get("audio", {}).get("n_fft"),
                "hop_length": CONFIG.get("audio", {}).get("hop_length"),
                "lr_initial": CONFIG.get("training", {}).get("learning_rate"),
                "batch_size": CONFIG.get("training", {}).get("batch_size"),
                "seed": CONFIG.get("random_seed"),
                "calibration_mode": CONFIG.get("mode", {}).get("calibration_mode", False)
            }
            params_to_log_filtered = {k: v for k, v in params_to_log.items() if v is not None}
            mlflow.log_params(params_to_log_filtered)
            print("Начальный конфиг и параметры залогированы в MLflow.")
        except Exception as e_mlflow_start:
            print(f"⚠️ Ошибка при старте или логировании в MLflow: {e_mlflow_start}")
            print("   MLflow будет считаться неактивным для этого запуска.")
            IS_MLFLOW_ACTIVE = False
            active_mlflow_run_id = None
    else:
        print("\nMLflow неактивен, логирование пропускается.")

except Exception as e_prep:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА на этапе подготовки к запуску: {e_prep}")
    traceback.print_exc(limit=2)
    if active_mlflow_run_id and mlflow.active_run() and mlflow.active_run().info.run_id == active_mlflow_run_id:
        mlflow.end_run(status="FAILED")
        print("MLflow run завершен со статусом FAILED.")
    raise

print("\n--- Ячейка 8.1 (Подготовка и Старт MLflow) завершена ---")


 ЗАПУСК ОСНОВНОГО КОНВЕЙЕРА (Режим: train_and_finetune) 
 Директория для результатов этого запуска: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\outputs\CRNN_ResNetSE_K3x5-K3x5_Hop96_v10_Base
Выходная директория ЗАПУСКА создана/проверена.
Начальная конфигурация ЗАПУСКА сохранена: config_initial_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10_Base.json

--- Старт MLflow Run ---
MLflow Run начат. ID: 50ccdbbc466143cd81f2693d66985788
Начальный конфиг и параметры залогированы в MLflow.

--- Ячейка 8.1 (Подготовка и Старт MLflow) завершена ---


### Ячейка 8.2: Основное Обучение (`train`)

Запускаем этап обучения с нуля, если `run_mode` это позволяет. Используем функцию `run_training` из `src`.

In [9]:
best_train_model_path = ""
best_train_lev = float('inf')
train_stage_executed = False

if run_mode in ["train_and_finetune", "train_only"]:
    print("\n" + "="*20 + " Этап: Основное Обучение ('train') " + "="*20)
    stage_start_time = time.time()
    train_stage_executed = True
    try:
        # Вызываем run_training БЕЗ аргумента output_dir_run
        best_train_model_path, best_train_lev = run_training(
            mode='train',
            config=CONFIG,
            # output_dir_run=OUTPUT_DIR_RUN, # <--- УДАЛЕНО! Функция сама должна знать, куда сохранять, вероятно из CONFIG или project_root
            train_df=train_split_df,
            val_df=val_split_df,
            char_to_int=char_to_int,
            int_to_char=int_to_char,
            audio_augmenter_global=AUDIO_AUGMENTER_GLOBAL,
            available_map_indices=AVAILABLE_PERLIN_INDICES,
            checkpoint_path=None,
            IS_MLFLOW_ACTIVE=IS_MLFLOW_ACTIVE,
            project_root=PROJECT_ROOT # Оставляем, если функция run_training его ожидает (если нет - тоже убрать)
        )

        if best_train_model_path and np.isfinite(best_train_lev):
            final_model_path = best_train_model_path
            final_levenshtein = best_train_lev
            print(f"\n✅ Основное обучение ('train') завершено.")
            print(f"   Лучший Levenshtein (Val): {final_levenshtein:.4f}")
            print(f"   Модель сохранена: {Path(final_model_path).name}")
            if IS_MLFLOW_ACTIVE and active_mlflow_run_id:
                 with contextlib.suppress(Exception):
                      mlflow.log_metric("best_train_lev", final_levenshtein)
        else:
            print("\n⚠️ Основное обучение ('train') завершилось без сохранения лучшей модели.")

    except TypeError as te: # Ловим TypeError явно
        # Проверяем, не связана ли ошибка снова с project_root
        if 'project_root' in str(te):
             print(f"❌ ОШИБКА TypeError: Похоже, функция run_training() также не ожидает аргумент 'project_root'.")
             print("   Попробуйте убрать и его из вызова в Ячейке 8.2 и 8.3.")
        else:
             print(f"❌ ОШИБКА TypeError при вызове run_training(): {te}")
             print("   Проверьте сигнатуру функции run_training в src/training/pipeline.py и передаваемые аргументы.")
        traceback.print_exc(limit=1)
        run_mode = "ERROR_DURING_TRAIN" # Устанавливаем ошибку
        # Завершаем MLflow run, если он был начат
        if IS_MLFLOW_ACTIVE and active_mlflow_run_id and mlflow.active_run() and mlflow.active_run().info.run_id == active_mlflow_run_id:
            mlflow.log_metric("error_stage", 1)
            mlflow.end_run(status="FAILED")
            print("MLflow run завершен со статусом FAILED.")
            active_mlflow_run_id = None
        raise # Перевыбрасываем ошибку, чтобы остановить выполнение ячейки

    except Exception as e_train:
        print(f"❌ КРИТИЧЕСКАЯ ОШИБКА во время основного обучения ('train'): {e_train}")
        traceback.print_exc(limit=2)
        run_mode = "ERROR_DURING_TRAIN"
        print("!!! Обучение прервано. Следующие шаги будут пропущены.")
        if IS_MLFLOW_ACTIVE and active_mlflow_run_id and mlflow.active_run() and mlflow.active_run().info.run_id == active_mlflow_run_id:
            mlflow.log_metric("error_stage", 1)
            mlflow.end_run(status="FAILED")
            print("MLflow run завершен со статусом FAILED.")
            active_mlflow_run_id = None

    stage_duration = time.time() - stage_start_time
    print(f"--- Этап 'train' занял: {stage_duration:.2f} сек. ---")

elif run_mode == "finetune_only":
    # Логика для finetune_only остается без изменений...
    print("\n--- Этап: Основное Обучение ('train') ПРОПУЩЕН (режим finetune_only) ---")
    ft_only_path_str = CONFIG.get("finetuning", {}).get("finetune_only_checkpoint_path")
    if not ft_only_path_str:
        raise ValueError("В режиме 'finetune_only' не указан путь 'finetuning.finetune_only_checkpoint_path' в CONFIG!")
    ft_only_file = Path(ft_only_path_str)
    if not ft_only_file.is_absolute(): ft_only_file = PROJECT_ROOT / ft_only_path_str
    if not ft_only_file.exists(): raise FileNotFoundError(f"Не найден файл модели для finetune_only: {ft_only_file.resolve()}")

    final_model_path = str(ft_only_file.resolve())
    try:
        match = re.search(r"lev([\d.]+)\.pth", Path(final_model_path).name)
        final_levenshtein = float(match.group(1)) if match else float('inf')
    except Exception: final_levenshtein = float('inf')
    print(f"Будет использоваться модель для дообучения: {Path(final_model_path).name}")
    print(f"  (Предполагаемый Lev из имени файла: {final_levenshtein:.4f if np.isfinite(final_levenshtein) else 'N/A'})")
else:
    print(f"\n--- Этап: Основное Обучение ('train') ПРОПУЩЕН (Режим: {run_mode}) ---")

print(f"\n--- Ячейка 8.2 (Основное Обучение) завершена ---")


==================== Этап: Основное Обучение ('train') ====================

>> Запуск run_training (mode='train') <<

--- Инициализация параметров для этапа 'TRAIN' ---
Предупреждение: Путь к директории запуска не найден в config['execution_info'], используется: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\outputs\CRNN_ResNetSE_K3x5-K3x5_Hop96_v10_Base
  Device: cuda
  Output Dir (для чекпоинтов этапа): C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\outputs\CRNN_ResNetSE_K3x5-K3x5_Hop96_v10_Base

MLflow: Логирование для этапа 'train' активно.

Инициализация модели...

--- Инициализация ResNet-SE CNN слоев (из architectures.py) ---
  Начальная F=257, SE=True (ratio=16)
  Block 1: ResBlock(1, 32, k=(5, 5), s=(2, 2), se=True)
  Block 2: ResBlock(32, 64, k=(5, 5), s=(2, 2), se=True)
CNN: Общий фактор сжатия: Время=4, Частота=4
--- Инициализация RNN ---
  BiGRU: Input=4160, Hidden=128, Layers=2, Dropout=0.20
--- Инициализация Классификатора ---
  Dropout: 0.20
  Linear: 

Эпоха 1/20 [TRAIN]:   0%|                                                                                     …


❗️ Обучение (TRAIN) прервано пользователем (KeyboardInterrupt).

Очистка ресурсов после 'train'...
  GC собрал 1700 объектов.
  Кэш CUDA очищен.
Очистка ресурсов завершена.

>> Завершение run_training (mode='train').
   Возвращаемый путь (лучшая модель этапа): 'N/A'
   Возвращаемая метрика (лучший Val Levenshtein): inf

⚠️ Основное обучение ('train') завершилось без сохранения лучшей модели.
--- Этап 'train' занял: 10.05 сек. ---

--- Ячейка 8.2 (Основное Обучение) завершена ---


### Ячейка 8.3: Дообучение (`finetune`)
Запускаем этап дообучения, если условия позволяют. Используем ту же функцию `run_training` из `src`, но с другими параметрами (`mode='finetune'`, `checkpoint_path`).

In [ ]:
# Ячейка 8.3: Дообучение ('finetune')
# ----------------------------------

best_ft_model_path = ""
best_ft_lev = float('inf')
finetune_stage_executed = False

should_run_finetuning = (
    run_mode in ["train_and_finetune", "finetune_only"] and
    finetune_apply_flag and
    final_model_path and
    Path(final_model_path).exists()
)

if should_run_finetuning:
    print("\n" + "="*20 + " Этап: Дообучение ('finetune') " + "="*20)
    stage_start_time = time.time()
    finetune_stage_executed = True
    print(f"Используется базовая модель: {Path(final_model_path).name}")
    print(f"  Ее Levenshtein (Val) до начала FT: {final_levenshtein:.4f if np.isfinite(final_levenshtein) else 'N/A'}")

    try:
        best_ft_model_path, best_ft_lev = run_training(
            mode='finetune',
            config=CONFIG,
            output_dir_run=OUTPUT_DIR_RUN,
            train_df=train_split_df,
            val_df=val_split_df,
            char_to_int=char_to_int,
            int_to_char=int_to_char,
            audio_augmenter_global=AUDIO_AUGMENTER_GLOBAL,
            available_map_indices=AVAILABLE_PERLIN_INDICES,
            checkpoint_path=final_model_path, # Передаем базовую модель!
            IS_MLFLOW_ACTIVE=IS_MLFLOW_ACTIVE,
            project_root=PROJECT_ROOT
        )

        if best_ft_model_path and np.isfinite(best_ft_lev):
            print(f"\n✅ Дообучение ('finetune') завершено.")
            print(f"   Лучший Levenshtein FT: {best_ft_lev:.4f}")
            print(f"   Модель FT сохранена: {Path(best_ft_model_path).name}")
            if IS_MLFLOW_ACTIVE and active_mlflow_run_id:
                 with contextlib.suppress(Exception):
                      mlflow.log_metric("best_finetune_lev", best_ft_lev)

            if best_ft_lev < final_levenshtein: # Обновляем, только если стало лучше
                print(f"   Метрика УЛУЧШИЛАСЬ ({final_levenshtein:.4f if np.isfinite(final_levenshtein) else 'N/A'} -> {best_ft_lev:.4f}). Обновляем финальную модель.")
                final_model_path = best_ft_model_path
                final_levenshtein = best_ft_lev
            else:
                print(f"   Метрика НЕ УЛУЧШИЛАСЬ. Финальной остается предыдущая лучшая модель.")
        else:
            print("\n⚠️ Дообучение ('finetune') завершилось без сохранения лучшей модели.")

    except Exception as e_finetune:
        print(f"❌ КРИТИЧЕСКАЯ ОШИБКА во время дообучения ('finetune'): {e_finetune}")
        traceback.print_exc(limit=2)
        run_mode = "ERROR_DURING_FINETUNE"
        print("!!! Дообучение прервано.")
        if IS_MLFLOW_ACTIVE and active_mlflow_run_id and mlflow.active_run() and mlflow.active_run().info.run_id == active_mlflow_run_id:
            mlflow.log_metric("error_stage", 2)
            mlflow.end_run(status="FAILED")
            print("MLflow run завершен со статусом FAILED.")
            active_mlflow_run_id = None

    stage_duration = time.time() - stage_start_time
    print(f"--- Этап 'finetune' занял: {stage_duration:.2f} сек. ---")

elif run_mode not in ["ERROR_DURING_TRAIN"]:
     print("\n--- Этап: Дообучение ('finetune') ПРОПУЩЕН ---")
     if run_mode not in ["train_and_finetune", "finetune_only"]: print(f"   Причина: Режим '{run_mode}' не предполагает дообучение.")
     elif not finetune_apply_flag: print("   Причина: Флаг 'finetuning.apply' = False.")
     elif not final_model_path: print("   Причина: Нет базовой модели.")
     elif not Path(final_model_path).exists(): print(f"   Причина: Файл базовой модели не найден: {final_model_path}")
     else: print(f"   Причина: Неизвестная.")

print(f"\n--- Ячейка 8.3 (Дообучение) завершена ---")


--- Этап: Дообучение ('finetune') ПРОПУЩЕН ---
   Причина: Нет базовой модели.

--- Ячейка 8.3 (Дообучение) завершена ---


### Ячейка 8.4: Генерация Финального Submission
Генерируем файл `submission.csv` с помощью функции `generate_submission` из `src`.

In [ ]:
# Ячейка 8.4: Генерация Финального Submission
# -------------------------------------------

submission_generated_flag = False
submission_file_name = "submission_default.csv"
submission_save_path = None

if "ERROR" not in run_mode and final_model_path and Path(final_model_path).exists():
    print("\n" + "="*20 + " Этап: Генерация Финального Submission " + "="*20)
    stage_start_time = time.time()
    print(f"Используется финальная лучшая модель: {Path(final_model_path).name}")
    print(f"  Ее лучший Val Levenshtein: {final_levenshtein:.4f if np.isfinite(final_levenshtein) else 'N/A'}")

    mode_suffix = run_mode if "ERROR" not in run_mode else "error_run"
    lev_suffix = f"lev{final_levenshtein:.4f}" if np.isfinite(final_levenshtein) else "levNA"
    submission_file_name = f"submission_{run_dir_name}_{mode_suffix}_{lev_suffix}.csv"
    submission_save_path = OUTPUT_DIR_RUN / submission_file_name

    try:
        submission_generated_flag = generate_submission(
            config=CONFIG,
            model_path=final_model_path,
            test_df=test_df,
            char_to_int=char_to_int,
            int_to_char=int_to_char,
            output_dir_run=OUTPUT_DIR_RUN,
            submission_filename=submission_file_name,
            IS_MLFLOW_ACTIVE=IS_MLFLOW_ACTIVE,
            project_root=PROJECT_ROOT
        )
        if submission_generated_flag:
            print(f"✅ Файл Submission успешно сгенерирован: {submission_save_path.resolve()}")
            if IS_MLFLOW_ACTIVE and active_mlflow_run_id:
                 with contextlib.suppress(Exception):
                    mlflow.log_artifact(str(submission_save_path), artifact_path="submissions")
                    print("Файл Submission залогирован как артефакт в MLflow.")
        else:
            print("⚠️ Функция generate_submission сообщила об ошибке во время генерации.")

    except NameError as ne:
        print(f"❌ ОШИБКА: Переменная 'test_df' не определена. Выполните Ячейку 5.")
        traceback.print_exc(limit=1)
    except Exception as e_sub:
        print(f"❌ КРИТИЧЕСКАЯ ОШИБКА во время генерации submission: {e_sub}")
        traceback.print_exc(limit=2)
        submission_generated_flag = False
        run_mode = "ERROR_DURING_SUBMISSION"
        if IS_MLFLOW_ACTIVE and active_mlflow_run_id and mlflow.active_run() and mlflow.active_run().info.run_id == active_mlflow_run_id:
            mlflow.log_metric("error_stage", 3)
            mlflow.end_run(status="FAILED")
            print("MLflow run завершен со статусом FAILED.")
            active_mlflow_run_id = None

    stage_duration = time.time() - stage_start_time
    print(f"--- Этап 'submission' занял: {stage_duration:.2f} сек. ---")

elif "ERROR" not in run_mode:
    print("\n--- Этап: Генерация Submission ПРОПУЩЕНА ---")
    if not final_model_path: print("   Причина: Нет финальной модели.")
    elif not Path(final_model_path).exists(): print(f"   Причина: Файл финальной модели не найден: {final_model_path}")
else:
     print(f"\n--- Этап: Генерация Submission ПРОПУЩЕНА из-за предыдущей ошибки ({run_mode}) ---")

print(f"\n--- Ячейка 8.4 (Генерация Submission) завершена ---")


--- Этап: Генерация Submission ПРОПУЩЕНА ---
   Причина: Нет финальной модели.

--- Ячейка 8.4 (Генерация Submission) завершена ---


### Ячейка 8.5: Завершение Пайплайна и MLflow

Сохраняем финальную конфигурацию с результатами и завершаем активный MLflow run.

In [ ]:
# Ячейка 8.5: Завершение Пайплайна и MLflow
# ----------------------------------------

print("\n" + "="*20 + " Этап: Завершение Пайплайна " + "="*20)

try:
    # --- Подготовка и сохранение финального конфига ---
    final_config_to_save = CONFIG.copy()
    pipeline_results = {
        "final_best_model_path_relative": str(Path(final_model_path).relative_to(PROJECT_ROOT)) if final_model_path and Path(final_model_path).is_file() else "N/A",
        "final_best_model_name": Path(final_model_path).name if final_model_path else "N/A",
        "final_best_val_levenshtein": final_levenshtein if np.isfinite(final_levenshtein) else -1.0,
        "executed_run_mode": run_mode,
        "train_stage_executed": train_stage_executed,
        "finetune_stage_executed": finetune_stage_executed,
        "submission_generated": submission_generated_flag,
        "submission_filename": submission_file_name if submission_generated_flag else "N/A",
        "submission_path_relative": str(submission_save_path.relative_to(PROJECT_ROOT)) if submission_save_path and submission_save_path.is_file() else "N/A",
        "num_available_perlin_maps": len(AVAILABLE_PERLIN_INDICES),
        "total_pipeline_duration_sec": round(time.time() - pipeline_start_time, 2),
        "mlflow_run_id": active_mlflow_run_id if active_mlflow_run_id else "N/A"
    }
    final_config_to_save["pipeline_results"] = pipeline_results

    final_config_filename = f"config_final_{run_dir_name}.json"
    final_config_save_path = OUTPUT_DIR_RUN / final_config_filename
    with open(final_config_save_path, 'w', encoding='utf-8') as f:
        json.dump(final_config_to_save, f, indent=4, ensure_ascii=False, default=str)
    print(f"Финальная конфигурация с результатами сохранена: {final_config_save_path.resolve()}")

    # --- Логирование в MLflow (если активен) ---
    if IS_MLFLOW_ACTIVE and active_mlflow_run_id and mlflow.active_run() and mlflow.active_run().info.run_id == active_mlflow_run_id:
         try:
             mlflow.log_artifact(str(final_config_save_path), artifact_path="config")
             print("Финальный конфиг залогирован как артефакт в MLflow.")
             if np.isfinite(final_levenshtein):
                 mlflow.log_metric("best_final_lev", final_levenshtein)
                 print(f"Финальная метрика best_final_lev ({final_levenshtein:.4f}) залогирована.")
         except Exception as e_art_cfg:
             print(f"Предупреждение: Не удалось залогировать финальный конфиг/метрику в MLflow: {e_art_cfg}")

except Exception as e_final_cfg:
    print(f"❌ Ошибка сохранения/логирования финального конфига: {e_final_cfg}")
    traceback.print_exc(limit=2)

# --- Завершение MLflow run ---
final_mlflow_status = "FINISHED" if "ERROR" not in run_mode else "FAILED"
if IS_MLFLOW_ACTIVE and active_mlflow_run_id and mlflow.active_run() and mlflow.active_run().info.run_id == active_mlflow_run_id:
    print(f"\nЗавершение финального MLflow run (ID: {active_mlflow_run_id}) со статусом {final_mlflow_status}...")
    mlflow.end_run(status=final_mlflow_status)
    print("MLflow run завершен.")
elif active_mlflow_run_id:
     print("\nMLflow run был завершен ранее (вероятно, из-за ошибки).")
else:
     print("\nMLflow не был активен в этом запуске.")

# --- Финальный вывод ---
pipeline_end_time = time.time()
total_pipeline_duration = pipeline_end_time - pipeline_start_time
print("\n" + "="*50 + f"\n КОНВЕЙЕР ЗАВЕРШЕН (Итоговый статус: {run_mode}) \n" + "="*50)
print(f"Итоговое время выполнения: {total_pipeline_duration:.2f} сек. ({total_pipeline_duration/60:.2f} мин.)")

if final_model_path and np.isfinite(final_levenshtein) and "ERROR" not in run_mode:
    print(f"\nФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:")
    print(f"  Лучшая модель: {Path(final_model_path).name}")
    print(f"  Лучший Levenshtein (Val): {final_levenshtein:.4f}")
    if submission_generated_flag: print(f"  Файл Submission: {submission_file_name} (в папке {OUTPUT_DIR_RUN.name})")
    else: print(f"  Файл Submission: НЕ СГЕНЕРИРОВАН или ошибка генерации")
elif "ERROR" in run_mode:
     print("\nПайплайн завершился с ОШИБКОЙ.")
     print(f"  Статус ошибки: {run_mode}")
     print(f"  Проверьте логи выше для деталей.")
else:
    print("\nФинальная модель не была успешно создана или сохранена.")

print("\n--- Ячейка 8.5 (Завершение Пайплайна) завершена ---")


==================== Этап: Завершение Пайплайна ====================
Финальная конфигурация с результатами сохранена: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\outputs\CRNN_ResNetSE_K3x5-K3x5_Hop96_v10_Base\config_final_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10_Base.json
Финальный конфиг залогирован как артефакт в MLflow.

Завершение финального MLflow run (ID: 30cbe9908ffa4529bed383c7f044e613) со статусом FINISHED...
MLflow run завершен.

 КОНВЕЙЕР ЗАВЕРШЕН (Итоговый статус: train_and_finetune) 
Итоговое время выполнения: 1.77 сек. (0.03 мин.)

Финальная модель не была успешно создана или сохранена.

--- Ячейка 8.5 (Завершение Пайплайна) завершена ---
